# Mount Google Drive (only for checkpoints, dataset comes from GitHub)
from google.colab import drive
import os

# Check if drive is already mounted, and handle accordingly
if os.path.exists('/content/drive/MyDrive'):
    print('✓ Google Drive is already mounted')
else:
    # Try to mount (force_remount will unmount first if needed)
    try:
        drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted successfully')
    except ValueError as e:
        if 'already contain files' in str(e):
            print('⚠ Drive mountpoint has files - attempting force remount...')
            drive.mount('/content/drive', force_remount=True)
            print('✓ Google Drive remounted successfully')
        else:
            raise

# Clone or update the repository (includes dataset in data/hit-uav/)
if os.path.exists('SGGF-Net'):
    print('\nRepository already exists, pulling latest changes...')
    %cd SGGF-Net
    !git pull origin main
else:
    !git clone https://github.com/HarishSankarK/SGGF-Net.git
    %cd SGGF-Net

# Verify dataset is included
print('\nVerifying dataset...')
!ls -la data/hit-uav/ 2>/dev/null && echo "✓ Dataset found in repository!" || echo "⚠ Dataset not found"


In [ ]:
# ============================================================================
# STEP 2-5: Setup, Verify Dataset, and Start Training
# ============================================================================

# Step 1: Verify we're in the right directory
import os
print("=" * 70)
print("STEP 1: Verifying directory...")
print("=" * 70)
print(f"Current directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

# Step 2: Setup GPU Runtime and Install Dependencies
print("\n" + "=" * 70)
print("STEP 2: Setting up GPU Runtime (T4)...")
print("=" * 70)
print("📋 Make sure GPU runtime is enabled:")
print("   1. Go to: Runtime → Change runtime type")
print("   2. Select: GPU (T4) - NOT TPU or CPU")
print("   3. Click: Save")
print("   4. Runtime → Restart runtime (if changing from TPU)")
print("")

# Install PyTorch and dependencies
print("Installing PyTorch and dependencies...")
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install other required dependencies
print("\nInstalling other dependencies...")
!pip install numpy pillow opencv-python tqdm matplotlib scipy

# Verify GPU setup
print("\nChecking GPU availability...")
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
    print(f'✓ GPU detected: {gpu_name}')
    print(f'✓ GPU Memory: {gpu_memory:.1f} GB')
    if 'T4' in gpu_name:
        print('✓ T4 GPU detected - optimal for this training!')
    else:
        print(f'⚠ GPU is {gpu_name} (not T4, but should work)')
    device_available = True
else:
    print('⚠ GPU not available - training will use CPU (very slow!)')
    print('  Make sure GPU runtime is enabled in Colab settings')
    device_available = False

if device_available:
    print('\n✅ GPU setup complete! Ready for training.')
else:
    print('\n⚠⚠⚠ WARNING: GPU NOT DETECTED ⚠⚠⚠')
    print('Training will use CPU, which is extremely slow!')
    print('Please enable GPU runtime before training.')

# Step 3: Verify Dataset
print("\n" + "=" * 70)
print("STEP 3: Verifying dataset...")
print("=" * 70)

if os.path.exists('data/hit-uav'):
    print("✓ Dataset found in data/hit-uav/")
    try:
        train_img_dir = 'data/hit-uav/images/train'
        train_label_dir = 'data/hit-uav/labels/train'
        if os.path.exists(train_img_dir):
            num_images = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"  Images: {num_images} training images")
        else:
            print(f"  ⚠ Images directory not found: {train_img_dir}")
            
        if os.path.exists(train_label_dir):
            num_labels = len([f for f in os.listdir(train_label_dir) if f.endswith('.txt')])
            print(f"  Labels: {num_labels} training labels")
        else:
            print(f"  ⚠ Labels directory not found: {train_label_dir}")
            
        print("\nDataset structure:")
        !ls -la data/hit-uav/
    except Exception as e:
        print(f"⚠ Error checking dataset: {e}")
        print("  But dataset directory exists, continuing...")
else:
    print("⚠ Dataset not found. Make sure you've pushed it to GitHub.")

# Step 4: Setup checkpoint directory and start training
print("\n" + "=" * 70)
print("STEP 4: Starting Training on T4 GPU...")
print("=" * 70)

# Setup checkpoint directory in Google Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(drive_checkpoint_dir, exist_ok=True)
print(f'Checkpoints will be saved to: {drive_checkpoint_dir}')

# Final GPU check before training
print('')
print('🔍 Final GPU check before training...')
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✓ GPU ready: {gpu_name}')
    print('✓ Training will use GPU (fast!)')
else:
    print('⚠⚠⚠ GPU NOT AVAILABLE! ⚠⚠⚠')
    print('')
    print('Training will use CPU, which will be EXTREMELY slow!')
    print('')
    print('📋 TO ENABLE GPU:')
    print('   1. Go to: Runtime → Change runtime type')
    print('   2. Select: GPU (T4)')
    print('   3. Click: Save')
    print('   4. Runtime → Restart runtime')
    print('   5. Re-run this entire cell (Cell 1)')
    print('')
    print('⚠ Continuing without GPU is NOT recommended!')
    print('   Press Ctrl+C to cancel, or wait 3 seconds to continue anyway...')
    import time
    try:
        time.sleep(3)
    except KeyboardInterrupt:
        print('\n✓ Training cancelled. Please enable GPU runtime first.')
        raise

# T4 GPU OPTIMIZED SETTINGS
# T4 GPU has 16GB VRAM - we use conservative settings to avoid OOM
# - Batch size: 1 (safe for T4, can try 2 if you have memory)
# - Max size: 800 (balanced for memory and performance)
# - Gradient accumulation: 4 (effective batch_size = 4)
# - Mixed Precision (AMP): Enabled (2x faster, uses less memory)
# - Learning rate: 0.0005 (stable for this batch size)

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION (T4 GPU Optimized):")
print("=" * 70)
print("Device: GPU (T4)")
print("Batch size: 1 (per GPU)")
print("Gradient accumulation: 4 (effective batch_size = 4)")
print("Max image size: 800")
print("Mixed Precision: Enabled (AMP)")
print("Learning rate: 0.0005")
print("=" * 70)
print("")

# Start training on GPU using train.py
!python scripts/train.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --num_classes 6 \
    --batch_size 1 \
    --grad_accum_steps 4 \
    --num_epochs 50 \
    --lr 0.0005 \
    --max_size 800 \
    --warmup_epochs 5 \
    --checkpoint_dir {drive_checkpoint_dir} \
    --device cuda \
    --use_amp \
    --val_freq 10

print("\n" + "=" * 70)
print("Training completed!")
print("=" * 70)


## Step 6: Evaluate Model

Evaluate the trained model on test/validation set.


In [ ]:
# Evaluate on test/validation set (using checkpoint from Drive)
import os
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'

# Choose checkpoint to evaluate (prefer best.pth, fallback to latest.pth)
if os.path.exists(best_checkpoint):
    checkpoint_path = best_checkpoint
    print(f'✓ Using best checkpoint: {checkpoint_path}')
elif os.path.exists(latest_checkpoint):
    checkpoint_path = latest_checkpoint
    print(f'⚠ Best checkpoint not found, using latest: {checkpoint_path}')
else:
    print('❌ No checkpoint found!')
    print(f'   Checkpoint directory: {drive_checkpoint_dir}')
    print('   Please train the model first (run Cell 1)')
    raise FileNotFoundError(f'No checkpoint found in {drive_checkpoint_dir}')

print('')
print('='*70)
print('EVALUATION CONFIGURATION (T4 GPU Optimized):')
print('='*70)
print(f'Checkpoint: {checkpoint_path}')
print('Dataset: HIT-UAV')
print('Split: test (change to "val" for validation set)')
print('Batch size: 1 (T4 GPU optimized)')
print('Max image size: 800 (matches training)')
print('='*70)
print('')

# Evaluate on test set
!python scripts/evaluate.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --checkpoint {checkpoint_path} \
    --num_classes 6 \
    --batch_size 1 \
    --max_size 800 \
    --split test \
    --device cuda

print('')
print('='*70)
print('Evaluation completed!')
print('='*70)

# Optional: Also evaluate on validation set
print('')
print('='*70)
print('OPTIONAL: Evaluate on Validation Set')
print('='*70)
print('Uncomment and run the code below to also evaluate on validation set:')
print('')
# Uncomment the following lines to evaluate on validation set:
# !python scripts/evaluate.py \
#     --dataset hituav \
#     --data_dir data/hit-uav \
#     --checkpoint {checkpoint_path} \
#     --num_classes 6 \
#     --batch_size 1 \
#     --max_size 800 \
#     --split val \
#     --device cuda


## Step 6B: Evaluate on Validation Set

If you want to evaluate on the validation set separately:


In [ ]:
# Evaluate on validation set (using checkpoint from Drive)
import os
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'

# Choose checkpoint to evaluate (prefer best.pth, fallback to latest.pth)
if os.path.exists(best_checkpoint):
    checkpoint_path = best_checkpoint
    print(f'✓ Using best checkpoint: {checkpoint_path}')
elif os.path.exists(latest_checkpoint):
    checkpoint_path = latest_checkpoint
    print(f'⚠ Best checkpoint not found, using latest: {checkpoint_path}')
else:
    print('❌ No checkpoint found!')
    print(f'   Checkpoint directory: {drive_checkpoint_dir}')
    print('   Please train the model first (run Cell 1)')
    raise FileNotFoundError(f'No checkpoint found in {drive_checkpoint_dir}')

print('')
print('='*70)
print('EVALUATION ON VALIDATION SET (T4 GPU Optimized):')
print('='*70)
print(f'Checkpoint: {checkpoint_path}')
print('Dataset: HIT-UAV')
print('Split: val (validation set)')
print('Batch size: 1 (T4 GPU optimized)')
print('Max image size: 800 (matches training)')
print('='*70)
print('')

# Evaluate on validation set
!python scripts/evaluate.py \
    --dataset hituav \
    --data_dir data/hit-uav \
    --checkpoint {checkpoint_path} \
    --num_classes 6 \
    --batch_size 1 \
    --max_size 800 \
    --split val \
    --device cuda

print('')
print('='*70)
print('Validation evaluation completed!')
print('='*70)


## Step 7: Resume Training from Drive Checkpoint


In [ ]:
# Resume training from a checkpoint saved in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'

# Check if checkpoint exists
import os
latest_checkpoint = f'{drive_checkpoint_dir}/latest.pth'
best_checkpoint = f'{drive_checkpoint_dir}/best.pth'

if os.path.exists(latest_checkpoint):
    resume_from = latest_checkpoint
    print(f'Resuming from: {resume_from}')
elif os.path.exists(best_checkpoint):
    resume_from = best_checkpoint
    print(f'Resuming from: {resume_from}')
else:
    resume_from = None
    print('No checkpoint found, starting fresh training')

# Resume training on GPU (using GPU-optimized settings)
# All optimizations applied: LR=0.0005, max_size=800, patch_size=32, batch_size=1 for GPU
if resume_from:
    !python scripts/train.py \
        --dataset hituav \
        --data_dir data/hit-uav \
        --num_classes 6 \
        --batch_size 1 \
        --grad_accum_steps 1 \
        --num_epochs 50 \
        --lr 0.0005 \
        --max_size 800 \
        --warmup_epochs 5 \
        --checkpoint_dir {drive_checkpoint_dir} \
        --resume {resume_from} \
        --device cuda \
        --val_freq 10 \
        --use_amp
else:
    print('No checkpoint to resume from. Run Step 5 to start training.')


## Checkpoint Management

Checkpoints are automatically saved to Google Drive at:
`/content/drive/MyDrive/SGGF-Net-checkpoints/`

- `latest.pth` - Latest checkpoint (every epoch)
- `best.pth` - Best model based on mAP

These persist even after Colab session ends!


In [ ]:
# List checkpoints in Drive
drive_checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
import os

if os.path.exists(drive_checkpoint_dir):
    print(f"Checkpoints in Drive ({drive_checkpoint_dir}):")
    checkpoints = os.listdir(drive_checkpoint_dir)
    for ckpt in checkpoints:
        if ckpt.endswith('.pth'):
            size = os.path.getsize(f'{drive_checkpoint_dir}/{ckpt}') / (1024*1024)  # MB
            print(f"  - {ckpt} ({size:.2f} MB)")
else:
    print(f"Checkpoint directory not found: {drive_checkpoint_dir}")
    print("Run Step 5 to start training and create checkpoints.")
